# 01 - Tiền xử lý Dữ liệu ViHOS & First-token Subword Alignment
**Môn học:** Xử lý Ngôn ngữ Tự nhiên - ĐH Công nghệ Thông tin (UIT)
**Nhân sự phụ trách:** Nông Nguyễn Thành (26410115)


In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer


## 1. Nạp dữ liệu ViHOS Benchmark


In [ ]:
train_path = "../data/processed/train.json"
dev_path = "../data/processed/dev.json"
test_path = "../data/processed/test.json"

with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

print(f"Tổng số mẫu train: {len(train_data)}")
print("Ví dụ mẫu đầu tiên:", train_data[0])


## 2. Thống kê phân bố nhãn BIO (O, B-HOS, I-HOS)


In [ ]:
all_tags = [t for sample in train_data for t in sample['tags']]
tag_counts = pd.Series(all_tags).value_counts()
print("Phân bố nhãn BIO:\n", tag_counts)

plt.figure(figsize=(6, 3))
sns.barplot(x=tag_counts.index, y=tag_counts.values, palette="viridis")
plt.title("Phân bố nhãn BIO trong tập huấn luyện ViHOS")
plt.ylabel("Số lượng token")
plt.show()


## 3. Khảo sát phân bố độ dài câu (Sentence Length Distribution)


In [ ]:
lengths = [len(sample['tokens']) for sample in train_data]
print(f"Độ dài trung bình: {np.mean(lengths):.1f} từ")
print(f"Độ dài tối đa: {np.max(lengths)} từ")
print(f"Tỷ lệ câu <= 128 từ: {np.mean(np.array(lengths) <= 128) * 100:.2f}%")


## 4. Minh họa First-token Subword Alignment với PhoBERT


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

sample = train_data[0]
tokens = sample['tokens']
tags = sample['tags']

subwords_list = []
aligned_tags = []

for word, tag in zip(tokens, tags):
    subwords = tokenizer.tokenize(word)
    subwords_list.extend(subwords)
    aligned_tags.append(tag)
    for _ in subwords[1:]:
        aligned_tags.append("-100 (Ignored)")

for sub, tag in zip(subwords_list, aligned_tags):
    print(f"{sub:<15} --> {tag}")
